# ***Conditional Chain***

In [40]:
from  langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv

# This function will load all the variables from the .env file and will 
# make them available in the os.environ dictionary (env variables)
load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

Bro API KEY Variable exists


In [41]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

# Chain With Conditional Chains

In [43]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative : {input}")])



In [44]:
# TASK - 2 [LLM]

llm_structured_output = llm_openai.with_structured_output(llm_schema)



In [45]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)-> str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)



### Conditional Chain 1

In [46]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser

In [47]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

### Conditional Chain 2


In [48]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


### Final Orchestrator

In [49]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
     insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain


In [50]:
final_orchestrator.invoke({"input": "I loved this KGF movie"})

/Users/pratappadhy/Documents/Projects/Python/Pratap_Agentic_AI/2-LangChain-Basics/2.8-LangChain_Tutorial/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(movie_summary_flag='positive'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


'Positivity is a choice—and one that pays dividends at work and in life.\n\nThis week I focused on celebrating small wins: a solved bug, a clear meeting agenda, a teammate who stepped up. Those moments don’t just feel good — they build momentum, strengthen collaboration, and make challenging projects more manageable. A positive mindset doesn’t ignore problems; it frames them as opportunities to learn and improve.\n\nTry this: at the end of each day, note one thing that went well and one lesson learned. Share it with your team or keep it in a personal log. Over time those entries become a powerful record of progress.\n\nHow do you cultivate positivity at work? I’d love to hear your approaches and tips. #Leadership #Mindset #Teamwork #Gratitude'